# Analisis de conductores asistido por ChatGPT

## Objetivo
Calcular las horas y millas acumuladas de cada conductor y mostrar los diez con mas millas. Los archivos de entrada son `data/drivers.csv` y `data/timesheet.csv`.

## Propuesta de prompt
> Escribe codigo Python con Pandas para leer drivers.csv y timesheet.csv. Agrupa por driverId y suma solamente hours-logged y miles-logged. Une los totales con el nombre del conductor, valida que la relacion sea uno a uno y ordena por driverId. Exporta summary.csv sin indice. Genera un grafico horizontal de los diez conductores con mas millas; resuelve empates por driverId. Comprueba que los totales coincidan con el archivo original y excluye ssn y location de los resultados.

## Revision de la solucion
La agregacion debe sumar medidas, no el numero de semana. La union debe conservar todos los conductores con registros. Las rutas deben funcionar desde notebooks/ y desde la raiz del repositorio. Las siguientes celdas implementan y verifican esta propuesta; el prompt es una guia reproducible, no un registro de una demostracion de clase.


In [ ]:
# Preparacion y rutas de la actividad
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import pandas as pd

# Permite ejecutar desde notebooks/, la actividad o la raiz del repositorio.
activity_dir = next(
    path for path in (
        Path.cwd(), Path.cwd().parent, Path.cwd() / "P105_drivers_chatgpt"
    )
    if path.name == "P105_drivers_chatgpt" and (path / "data" / "drivers.csv").is_file()
)
submission_dir = activity_dir / "submission"
submission_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# Carga de los conductores
drivers = pd.read_csv(activity_dir / "data" / "drivers.csv")
drivers.head()


In [ ]:
# Carga de los registros semanales
timesheet = pd.read_csv(activity_dir / "data" / "timesheet.csv")
timesheet.head()


In [ ]:
# Promedio de horas y millas por conductor
metrics = ["hours-logged", "miles-logged"]
mean_timesheet = timesheet.groupby("driverId")[metrics].mean()
mean_timesheet.head()


In [ ]:
# Totales de horas y millas por conductor: week no es una medida sumable.
sum_timesheet = timesheet.groupby("driverId", as_index=False)[metrics].sum()
sum_timesheet.head()


In [ ]:
# Incorporar el nombre verificando que cada conductor sea unico.
summary = (
    sum_timesheet.merge(
        drivers[["driverId", "name"]], on="driverId", validate="one_to_one"
    )
    .sort_values("driverId")
    .reset_index(drop=True)
)
summary


In [ ]:
# Exportar la tabla sin una columna adicional de indice.
summary.to_csv(submission_dir / "summary.csv", index=False)


In [ ]:
# Los diez conductores con mas millas acumuladas.
# El identificador resuelve empates de manera reproducible.
top10 = summary.sort_values(
    ["miles-logged", "driverId"], ascending=[False, True]
).head(10)
top10


In [ ]:
# Grafico horizontal del ranking.
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top10["name"], top10["miles-logged"], color="tab:orange", alpha=0.8)
ax.invert_yaxis()
ax.set_title("Top 10 conductores por millas recorridas")
ax.set_xlabel("Millas acumuladas")
ax.set_ylabel("Conductor")
ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(submission_dir / "top10_drivers.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)


In [ ]:
# Validaciones independientes de la apariencia del grafico.
assert summary.driverId.is_unique
assert len(summary) == timesheet.driverId.nunique()
assert set(summary.driverId) == set(timesheet.driverId)
assert summary.columns.tolist() == ["driverId", "hours-logged", "miles-logged", "name"]
assert summary[metrics].sum().equals(timesheet[metrics].sum())
assert summary["name"].notna().all()
assert len(top10) == min(10, len(summary))
assert top10["miles-logged"].is_monotonic_decreasing
pd.testing.assert_frame_equal(pd.read_csv(submission_dir / "summary.csv"), summary)
print(f"Validacion correcta: {len(summary)} conductores y {len(top10)} en el ranking.")


## Conclusion
El codigo propuesto debe comprobarse contra los datos originales. Las validaciones anteriores revisan la union, las columnas publicadas, los totales y el archivo exportado. El grafico y el CSV se guardan en submission/.
